In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
import os

# Now import UNET model and HabitatDataset class
from ImagePreparation import HabitatDataset, prepare_dataset
from UNETModel import UNET

# training the UNET model
# contains some LLM-generated code

In [2]:
# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Class weights for handling severe class imbalance
class_weights = torch.tensor([
    1/0.035, 1/0.0306, 1/0.0765, 1/0.1191,
    1/0.086, 1/0.3789, 1/0.2741
], dtype=torch.float32).to(device)

In [3]:
def train_fn(loader, model, optimizer, loss_fn, scaler):
    """Train for one epoch and return average loss."""
    model.train()
    loop = tqdm(loader, desc="Training")
    epoch_loss = 0.0
    num_batches = 0

    for img, mask in loop:
        img = img.to(device)
        mask = mask.to(device)

        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            preds = model(img)
            loss = loss_fn(preds, mask.long())

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item()
        num_batches += 1
        loop.set_postfix(loss=loss.item())
    
    return epoch_loss / num_batches


def test_fn(loader, model, loss_fn):
    """Evaluate model on test set and return average loss."""
    model.eval()
    loop = tqdm(loader, desc="Testing")
    epoch_loss = 0.0
    num_batches = 0
    
    with torch.no_grad():
        for img, mask in loop:
            img = img.to(device)
            mask = mask.to(device)
            
            preds = model(img)
            loss = loss_fn(preds, mask.long())
            
            epoch_loss += loss.item()
            num_batches += 1
            loop.set_postfix(loss=loss.item())
    
    return epoch_loss / num_batches

In [4]:
def plot_losses(train_losses, test_losses, save_path='loss_plot.png'):
    """Plot training and test losses."""
    plt.figure(figsize=(10, 6))
    epochs = range(1, len(train_losses) + 1)
    
    plt.plot(epochs, train_losses, 'b-', label='Training Loss', linewidth=2)
    plt.plot(epochs, test_losses, 'r-', label='Test Loss', linewidth=2)
    
    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('Loss', fontsize=12)
    plt.title('Training and Test Loss over Epochs', fontsize=14, fontweight='bold')
    plt.legend(fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"Loss plot saved to {save_path}")
    plt.show()


def print_results_summary(train_losses, test_losses, num_epochs):
    """Print a results summary paragraph."""
    print("\n" + "="*70)
    print("TRAINING RESULTS SUMMARY")
    print("="*70)
    
    print(f"\nThe U-Net model was trained for {num_epochs} epochs using Binary Cross-Entropy")
    print(f"loss with class weights to address data imbalance. The model was trained on")
    print(f"7 images and tested on 2 images from the benthic habitat dataset.")
    
    print(f"\nInitial Training Loss: {train_losses[0]:.4f}")
    print(f"Final Training Loss: {train_losses[-1]:.4f}")
    print(f"Training Loss Reduction: {((train_losses[0] - train_losses[-1]) / train_losses[0] * 100):.2f}%")
    
    print(f"\nInitial Test Loss: {test_losses[0]:.4f}")
    print(f"Final Test Loss: {test_losses[-1]:.4f}")
    print(f"Test Loss Reduction: {((test_losses[0] - test_losses[-1]) / test_losses[0] * 100):.2f}%")
    
    print(f"\nBest Training Loss: {min(train_losses):.4f} (Epoch {train_losses.index(min(train_losses)) + 1})")
    print(f"Best Test Loss: {min(test_losses):.4f} (Epoch {test_losses.index(min(test_losses)) + 1})")
    
    # Check for overfitting
    final_gap = train_losses[-1] - test_losses[-1]
    if abs(final_gap) < 0.1:
        print(f"\nThe model shows good generalization with minimal overfitting.")
    elif final_gap > 0.1:
        print(f"\nThe model shows signs of underfitting (test loss lower than train loss).")
    else:
        print(f"\nThe model shows some overfitting (train loss lower than test loss by {abs(final_gap):.4f}).")
    
    print("\n" + "="*70)


In [6]:
def main():
    # Configuration - optimized for ~30 minute training time
    IMG_DIR = r'C:\Users\Peter Chapman\Personal_Research_Projects\benthic-habitat-segmentation-ML\Project 4 Dataset\Project 4 Dataset\raw_labeled_data\images'
    MASK_DIR = r'C:\Users\Peter Chapman\Personal_Research_Projects\benthic-habitat-segmentation-ML\Project 4 Dataset\Project 4 Dataset\raw_labeled_data\annotations'
    NUM_EPOCHS = 10  # Reduced from 50 for faster training
    BATCH_SIZE = 16  # Increased for faster processing
    LEARNING_RATE = 1e-4
    PATCH_SIZE = 256  # Smaller patches for speed
    SAMPLES_PER_IMAGE = 15  # Fewer patches per image
    
    # Data augmentation
    train_transform = A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.2),
        ToTensorV2(),
    ])
    
    test_transform = A.Compose([
        ToTensorV2(),
    ])
    
    # Get list of all images first
    all_images = sorted([f for f in os.listdir(IMG_DIR) if f.endswith('.tif')])
    print(f"Total images available: {len(all_images)}")
    print(f"Images: {all_images}")
    
    # Create separate datasets for train and test with different images
    # Training: First 7 images
    train_dataset = HabitatDataset(
        img_dir=IMG_DIR,
        mask_dir=MASK_DIR,
        mode='patch',
        patch_size=PATCH_SIZE,
        samples_per_image=SAMPLES_PER_IMAGE,
        transform=train_transform
    )
    
    # Manually filter to only use first 7 images for training
    train_indices = []
    for idx, sample in enumerate(train_dataset.samples):
        img_name = sample['img_name']
        if img_name in all_images[:7]:  # First 7 images
            train_indices.append(idx)
    
    train_subset = Subset(train_dataset, train_indices)
    
    # Testing: Last 2 images
    test_dataset = HabitatDataset(
        img_dir=IMG_DIR,
        mask_dir=MASK_DIR,
        mode='patch',
        patch_size=PATCH_SIZE,
        samples_per_image=5,  # Very few patches for quick testing
        transform=test_transform
    )
    
    # Manually filter to only use last 2 images for testing
    test_indices = []
    for idx, sample in enumerate(test_dataset.samples):
        img_name = sample['img_name']
        if img_name in all_images[7:9]:  # Last 2 images
            test_indices.append(idx)
    
    test_subset = Subset(test_dataset, test_indices)
    
    print(f"\nTraining on images: {all_images[:7]}")
    print(f"Number of training patches: {len(train_subset)} (7 images × {SAMPLES_PER_IMAGE} patches)")
    print(f"\nTesting on images: {all_images[7:9]}")
    print(f"Number of test patches: {len(test_subset)}")
    
    # Create dataloaders
    train_loader = DataLoader(
        train_subset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )
    
    test_loader = DataLoader(
        test_subset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )
    
    print(f"\nNumber of training batches per epoch: {len(train_loader)}")
    print(f"Number of test batches: {len(test_loader)}")
    print(f"\nConfiguration for ~30 minute training:")
    print(f"  - {NUM_EPOCHS} epochs")
    print(f"  - Batch size: {BATCH_SIZE}")
    print(f"  - Patches per image: {SAMPLES_PER_IMAGE}")
    print(f"  - Estimated time per epoch: ~3 minutes")
    print(f"  - Total estimated time: ~30 minutes")
    
    # Initialize model
    model = UNET(in_channels=8, out_channels=7).to(device)
    print(f"\nU-Net model initialized with {sum(p.numel() for p in model.parameters()):,} parameters")
    
    # Loss function: CrossEntropyLoss (appropriate for 7-class segmentation)
    # Note: Binary Cross Entropy is only for 2 classes. For 7 classes, we use CrossEntropyLoss
    loss_fn = nn.CrossEntropyLoss(weight=class_weights)
    print(f"Using CrossEntropyLoss with class weights (appropriate for multi-class segmentation)")
    
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scaler = torch.cuda.amp.GradScaler()
    
    # Training loop
    train_losses = []
    test_losses = []
    
    print(f"\nStarting training for {NUM_EPOCHS} epochs...")
    print("="*70)
    
    for epoch in range(NUM_EPOCHS):
        print(f"\nEpoch {epoch + 1}/{NUM_EPOCHS}")
        print("-" * 70)
        
        # Train
        train_loss = train_fn(train_loader, model, optimizer, loss_fn, scaler)
        train_losses.append(train_loss)
        
        # Test
        test_loss = test_fn(test_loader, model, loss_fn)
        test_losses.append(test_loss)
        
        print(f"Epoch {epoch + 1} - Train Loss: {train_loss:.4f}, Test Loss: {test_loss:.4f}")
        
        # Save checkpoint every 10 epochs
        if (epoch + 1) % 10 == 0:
            checkpoint_path = f"unet_checkpoint_epoch_{epoch + 1}.pth"
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'train_loss': train_loss,
                'test_loss': test_loss,
            }, checkpoint_path)
            print(f"Checkpoint saved: {checkpoint_path}")
    
    # Save final model
    final_model_path = "unet_final_model.pth"
    torch.save(model.state_dict(), final_model_path)
    print(f"\nFinal model saved to {final_model_path}")
    
    # Save losses to file
    np.save('train_losses.npy', np.array(train_losses))
    np.save('test_losses.npy', np.array(test_losses))
    print("Losses saved to train_losses.npy and test_losses.npy")
    
    # Plot losses
    plot_losses(train_losses, test_losses)
    
    # Print results summary
    print_results_summary(train_losses, test_losses, NUM_EPOCHS)
    
    return model, train_losses, test_losses


if __name__ == "__main__":
    model, train_losses, test_losses = main()

Total images available: 9
Images: ['WV02052023.tif', 'WV02202022.tif', 'WV220101215.tif', 'WV220130115.tif', 'WV220140205.tif', 'WV220171020.tif', 'WV220230219.tif', 'WV320141230.tif', 'WV_01132022.tif']
Computing band statistics for normalization...
Band means: [333.43372 312.73245 366.93146 276.62698 170.5221  256.37604 304.4557
 236.65265]
Band stds: [106.48118  127.770424 205.53941  204.27007  144.62096  251.54451
 359.8682   283.61325 ]
Computing band statistics for normalization...
Band means: [333.43372 312.73245 366.93146 276.62698 170.5221  256.37604 304.4557
 236.65265]
Band stds: [106.48118  127.770424 205.53941  204.27007  144.62096  251.54451
 359.8682   283.61325 ]

Training on images: ['WV02052023.tif', 'WV02202022.tif', 'WV220101215.tif', 'WV220130115.tif', 'WV220140205.tif', 'WV220171020.tif', 'WV220230219.tif']
Number of training patches: 105 (7 images × 15 patches)

Testing on images: ['WV320141230.tif', 'WV_01132022.tif']
Number of test patches: 10

Number of traini

C:\RTemp\ipykernel_30764\1146446928.py:107: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()



U-Net model initialized with 31,040,903 parameters
Using CrossEntropyLoss with class weights (appropriate for multi-class segmentation)

Starting training for 10 epochs...

Epoch 1/10
----------------------------------------------------------------------


Training:   0%|          | 0/7 [00:00<?, ?it/s]C:\RTemp\ipykernel_30764\1155298996.py:13: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Testing: 100%|██████████| 1/1 [00:12<00:00, 12.25s/it, loss=1.95]


Epoch 1 - Train Loss: 1.6578, Test Loss: 1.9538

Epoch 2/10
----------------------------------------------------------------------


...

Testing: 100%|██████████| 1/1 [00:13<00:00, 13.58s/it, loss=1.3]


Epoch 9 - Train Loss: 0.9298, Test Loss: 1.2956

Epoch 10/10
----------------------------------------------------------------------


Testing: 100%|██████████| 1/1 [00:09<00:00,  9.41s/it, loss=0.89]


Epoch 10 - Train Loss: 0.8724, Test Loss: 0.8900
Checkpoint saved: unet_checkpoint_epoch_10.pth

Final model saved to unet_final_model.pth
Losses saved to train_losses.npy and test_losses.npy


: 